In [12]:
import pandas as pd
import json

In [13]:
happiness_df = pd.read_csv("happiness.csv")
steps_df = pd.read_csv("world_map_steps_average.csv")
wine_df = pd.read_csv("wine-consumption-per-capita.csv")
diet_df = pd.read_csv("world_diets.csv")
life_expectancy_df = pd.read_csv("life-expectancy.csv")

### Individual features

##### 1. Happiness (no ISO code)

In [14]:
happiness_df = happiness_df.rename(columns={"Life evaluation (3-year average)": "Life evaluation", "Country name": "country"})

In [15]:
#happiness_df["country"].to_csv("happiness_countries.csv", index=False)

##### 2. Steps (no ISO code)

In [16]:
steps_df = steps_df.rename(columns={"region": "country"})

In [17]:
#steps_df["country"].to_csv("steps_countries.csv", index=False)

##### 3. Wine consumption

In [18]:
wine_df = wine_df.rename(
    columns={
        "Alcohol, recorded per capita (15+) consumption (in litres of pure alcohol) - Beverage types: Wine": "Wine Consumption", 
        "Code": "ISO",
        "Entity": "country"
        }
    )

In [19]:
#wine_df["country"].to_csv("wine_countries.csv", index=False)

##### 4. Diet

In [20]:
diet_df = diet_df.rename(columns={"Entity": "country", "Code": "ISO"})

In [21]:
#https://www.who.int/news-room/fact-sheets/detail/healthy-diet

def rule80(calories):
    if calories < 1200:
        return calories / 1300 #not enough calories that's dangerous       
    elif calories <= 2100: # recommended calories intake
        return 1.0                    
    else:
        return max(0, 1 - (calories - 2100) / 2600)  #excessive calories intake


diet_df["rule80_score"] = diet_df["Calories intake"].apply(rule80)

In [22]:
#diet_df["country"].to_csv("diet_countries.csv", index=False)

##### 5. Life expectancy

In [23]:
life_expectancy_df = life_expectancy_df.rename(columns={"Entity": "country", "Code": "ISO"})

In [24]:
#life_expectancy_df["country"].to_csv("life_expectancy_countries.csv", index=False)

##### Merging the numeric codes dataframes with their corresponding dataframes

In [25]:
countries_id_json = pd.read_csv("numeric_codes/countries_id_json.csv")
#countries_id_json = countries_id_json.dropna(subset="id_json")

happiness_numeric_codes_df = pd.read_csv("numeric_codes/happiness_countries.csv")
steps_numeric_codes_df = pd.read_csv("numeric_codes/steps_countries.csv")
wine_numeric_codes_df = pd.read_csv("numeric_codes/wine_countries.csv")
diet_numeric_codes_df = pd.read_csv("numeric_codes/diet_countries.csv")
life_expectancy_numeric_codes_df = pd.read_csv("numeric_codes/life_expectancy_countries.csv")

In [26]:
happiness_df = pd.concat([happiness_df, happiness_numeric_codes_df["id_json"]], axis=1)
steps_df = pd.concat([steps_df, steps_numeric_codes_df["id_json"]], axis=1)
wine_df = pd.concat([wine_df, wine_numeric_codes_df["id_json"]], axis=1)
diet_df = pd.concat([diet_df, diet_numeric_codes_df["id_json"]], axis=1)
life_expectancy_df = pd.concat([life_expectancy_df, life_expectancy_numeric_codes_df["id_json"]], axis=1)

In [27]:
YEARS = list(range(1965, 2025))

h_y    = happiness_df[happiness_df["Year"].isin(YEARS)][["Year", "id_json", "Life evaluation"]].copy()
le_y   = life_expectancy_df[life_expectancy_df["Year"].isin(YEARS)][["Year", "id_json", "Life expectancy"]].copy()
diet_y = diet_df[diet_df["Year"].isin(YEARS)][["Year", "id_json", "plant_based_ratio", "rule80_score"]].copy()
wine_y = wine_df[wine_df["Year"].isin(YEARS)][["Year", "id_json", "Wine Consumption"]].copy()


### Blue Zone Index

In [28]:
all_countries = countries_id_json[["country_json", "id_json"]].drop_duplicates()
base = (
    pd.MultiIndex.from_product([all_countries["id_json"].tolist(), YEARS], names=["id_json", "Year"])
    .to_frame(index=False)
    .merge(all_countries, on="id_json", how="left")
)

# merged = (
#     base
#     .merge(le_y[["Year", "ISO", "Life expectancy"]],             on=["Year", "ISO"],     how="left")
#     .merge(diet_y[["Year", "ISO", "plant_based_ratio","rule80_score"]], on=["Year", "ISO"],   how="left")
#     .merge(wine_y[["Year", "ISO", "Wine Consumption"]],           on=["Year", "ISO"],     how="left")
#     .merge(steps_df[["country", "steps_mean_filtered"]],          on="country",           how="left")
#     .merge(h_y[["Year", "country", "Life evaluation"]],           on=["Year", "country"], how="left")
# )

merged = base\
    .merge(right=h_y, on=["Year", "id_json"], how="left")\
    .merge(right=steps_df[["steps_mean_filtered", "id_json"]], on="id_json", how="left")\
    .merge(right=wine_y, on=["Year", "id_json"], how="left")\
    .merge(right=diet_y, on=["Year", "id_json"], how="left")\
    .merge(right=le_y, on=["Year", "id_json"], how="left")

In [29]:
#we fill the missing data with the closest available data for this country in the closest year were data is available 
merged = merged.sort_values(["id_json", "Year"])
for col in ["Life expectancy", "plant_based_ratio", "rule80_score", "Wine Consumption", "Life evaluation"]:
    merged[col] = merged.groupby("id_json")[col].ffill()
    merged[col] = merged.groupby("id_json")[col].bfill()


In [30]:
merged["plant_based_ratio"]   = merged["plant_based_ratio"]   / merged["plant_based_ratio"].max()
merged["rule80_score"]   = merged["rule80_score"]   / merged["rule80_score"].max()
merged["Wine Consumption"]    = merged["Wine Consumption"]    / merged["Wine Consumption"].max()
merged["steps_mean_filtered"] = merged["steps_mean_filtered"] / merged["steps_mean_filtered"].max()
merged["Life evaluation"]     = merged["Life evaluation"]     / merged["Life evaluation"].max()

In [31]:
merged = merged.fillna(0)
merged = merged.rename(columns={"country_json": "country"})
merged["blue_zone_index"] = (
    merged["Life evaluation"] +
    merged["steps_mean_filtered"] +
    merged["Wine Consumption"] +
    merged["plant_based_ratio"]
) / 4

MAP_COLS     = ["blue_zone_index", "Life evaluation", "steps_mean_filtered",
                "Wine Consumption", "plant_based_ratio", "Life expectancy","rule80_score"]

map_out     = {}
scatter_out = {}


In [32]:
for year in YEARS:
    ydf = merged[merged["Year"] == year].copy()
    ydf = ydf.groupby(["id_json", "country"])[MAP_COLS].mean().reset_index()
    map_out[str(year)]     = ydf.set_index("id_json")[MAP_COLS].to_dict(orient="index")
    scatter_out[str(year)] = ydf[["id_json", "country"] + MAP_COLS].to_dict(orient="records")

with open("blue-zone-index-by-year.json", "w") as f:
    json.dump(map_out, f)

with open("blue-zone-index-scatter-plot-by-year.json", "w") as f:
    json.dump(scatter_out, f)


In [33]:
import json

with open('blue-zone-index-by-year.json', 'r') as f:
    data = json.load(f)

best_countries = {}

for year, countries in data.items():
    best_country = max(
        countries.items(), 
        key=lambda item: item[1].get('blue_zone_index', 0)
    )
    best_countries[year] = {
        "id": best_country[0],
        "score": best_country[1]['blue_zone_index']
    }

with open('best_countries.json', 'w') as f:
    json.dump(best_countries, f, indent=4)



In [34]:
countries_id_json = pd.read_csv("numeric_codes/countries_id_json.csv")
best_countries_df = pd.DataFrame.from_dict(best_countries, orient='index').reset_index()
best_countries_df = best_countries_df.rename(columns={"index": "year"})
best_countries_df["id"] = best_countries_df["id"].astype(int)
merged_winner = pd.merge(best_countries_df, countries_id_json, left_on="id", right_on="id_json")
merged_winner=merged_winner.sort_values(by="year",ascending=False)
merged_winner

,year,id,score,country_json,id_json
59,2024,752,0.631965,Sweden,752
58,2023,752,0.631933,Sweden,752
57,2022,752,0.631226,Sweden,752
56,2021,752,0.628324,Sweden,752
55,2020,756,0.630602,Switzerland,756
54,2019,756,0.630530,Switzerland,756
53,2018,756,0.625334,Switzerland,756
52,2017,756,0.623405,Switzerland,756
51,2016,756,0.624411,Switzerland,756
50,2015,756,0.624948,Switzerland,756
